# LoRA vs QLoRA 비교 학습 (Colab)

`training/tagging/train_lora.py` 를 **같은 인자로 `--load-in-4bit` 만 바꿔** 두 번 돌리고, 각 실행이 남긴
`run_metrics.json`(최대 GPU 메모리·학습 시간·비용·eval loss)을 표로 모은다. 학습 코드는 이 노트북에 없다 — 저장소 스크립트 그대로다.

| 순서 | 셀 | 하는 일 |
|---|---|---|
| 1 | GPU 확인 | **L4·A100 권장.** T4 는 BF16 이 없어 LoRA(16비트 원본)가 FP32 로 올라가 메모리가 넘친다 |
| 2 | ⚙️ 설정 | `TASK`, 데이터 위치, GPU 단가. **여기만 바꾼다** |
| 3 | 준비 | 드라이브 연결, 저장소 클론, `training/requirements.txt` 설치 |
| 4 | **시험 실행 (20 스텝)** | 두 모드를 짧게 돌려 **전체 학습 시간·비용·메모리를 미리 추정** → Colab 으로 갈지 RunPod 으로 갈지 정한다 |
| 5 | LoRA 학습 | `train_lora.py` — **먼저** (품질 기준점) |
| 6 | QLoRA 학습 | `train_lora.py --load-in-4bit` (나머지 인자는 5번과 같음) |
| 7 | 비교표 | 메모리·시간·비용·eval loss |
| 8 | 다음 단계 | vLLM 서빙과 품질 평가 (`training/RESULTS.md`) |

**학습 데이터는 맥에서 만들어 드라이브에 올린다** (`build_sft.py` 가 서빙 코드의 프롬프트를 import 하므로 `uv sync` 된 저장소에서 만든다).

```bash
# 태깅
uv run python training/tagging/from_teacher.py data/teacher/pilot200/tagging.jsonl --out data/tagging.jsonl
uv run python training/tagging/build_sft.py data/tagging.jsonl --out training/tagging/out --allow-unreviewed
# 추출
uv run python training/extraction/from_teacher.py data/teacher/pilot200/extraction.jsonl --out training/extraction/out
```
→ `training/tagging/out/`, `training/extraction/out/` 폴더(train.jsonl · val.jsonl · manifest.json)를 드라이브 `ktb-ai/sft/` 아래에 그대로 올린다.

> `--allow-unreviewed` 는 사람 검수 전 teacher 라벨로 **파이프라인을 시험할 때만** 쓴다. 이 노트북의 결과는 LoRA·QLoRA 비용 비교용이다.

## 1. GPU 확인

In [ ]:
import subprocess

import torch

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True
    ).stdout
)
if not torch.cuda.is_bf16_supported():
    print(
        "⚠️ BF16 미지원 GPU(T4 등). train_lora.py 는 이때 원본을 FP32 로 올리므로 LoRA 비교가 메모리 부족으로 실패할 수 있다."
    )
    print("   런타임 → 런타임 유형 변경에서 L4 또는 A100 을 고른다.")

## 2. ⚙️ 설정 — 이 셀만 바꾼다

| 값 | 뜻 |
|---|---|
| `TASK` | `"tagging"` 또는 `"extraction"` |
| `GPU_HOURLY_USD` | GPU 시간당 단가(달러). 비용 칸 계산용. 모르면 0 |
| `COMMON_ARGS` | 두 모드에 **똑같이** 들어가는 학습 인자. 비교 중에는 바꾸지 않는다 |

길이(`--max-length`)는 pilot200 최대 길이(태깅 975, 추출 3,257 토큰)에 여유를 둔 값이다.
두 모드 모두 gradient checkpointing 을 켜서 조건을 맞춘다.

In [ ]:
TASK = "tagging"  # "tagging" 또는 "extraction"
BRANCH = "feat/lena-lora-qlora-compare"
BASE_MODEL = "kakaocorp/kanana-2-3b-instruct"
REVISION = "6a5d7889964c4c590299d16e309eabab1f73f8a9"  # serving/vllm/config.yaml 과 같은 커밋
SFT_ROOT = "/content/drive/MyDrive/ktb-ai/sft"  # 그 아래 tagging/out, extraction/out
OUT_ROOT = "/content/drive/MyDrive/ktb-ai/adapters"
GPU_HOURLY_USD = 0.0  # 예: L4 약 0.7, A100 약 1.2 (클라우드 시세 기준. Colab 은 컴퓨팅 단위 환산)
SEED = 42

DATA_DIR = f"{SFT_ROOT}/{TASK}/out"
COMMON_ARGS = {
    "tagging": "--max-length 1280 --batch-size 8 --grad-accum 2",
    "extraction": "--max-length 4096 --batch-size 1 --grad-accum 16",
}[TASK] + f" --epochs 3 --learning-rate 2e-4 --lora-r 16 --lora-alpha 32 --seed {SEED} --gradient-checkpointing"
COMMON_ARGS += (
    f" --base-model {BASE_MODEL} --revision {REVISION} --data-dir {DATA_DIR} --gpu-hourly-usd {GPU_HOURLY_USD}"
)
print(TASK, DATA_DIR)
print(COMMON_ARGS)

## 3. 준비 — 드라이브 연결, 저장소 클론, 학습 의존성 설치

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
for name in ("train.jsonl", "val.jsonl", "manifest.json"):
    path = f"{DATA_DIR}/{name}"
    print(("✅ " if os.path.exists(path) else "❌ 없음: ") + path)

!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} https://github.com/100-hours-a-week/KTB4-17th-AI /content/repo
%cd /content/repo
!pip install -q -r training/requirements.txt

## 4. 시험 실행 — 20 스텝으로 전체 시간·비용·메모리 추정

두 모드를 20 스텝씩만 돌려 `run_metrics.json` 으로 전체 학습을 추정한다 (모델 다운로드·로드 시간은 빠진다).
첫 스텝은 준비 시간이 섞여 추정이 조금 크게 나온다 — 보수적인 값이다.

**Colab 으로 계속할지 RunPod 으로 갈지 (제안 기준)**

| 상황 | 고르기 |
|---|---|
| 한 번 학습이 2시간 안쪽이고 메모리가 남는다 | Colab 으로 계속 (세팅이 간단) |
| 한 번 학습이 3시간을 넘거나, 시드·재라벨링으로 여러 번 돌릴 예정 | RunPod (끊김·세션 제한 없음, 같은 GPU 를 시간 단위로 빌림) |
| 원하는 GPU(L4·A100)가 Colab 에서 안 잡힌다 | RunPod |

- Colab 비용은 오른쪽 위 **리소스 → 컴퓨팅 단위 사용률**에서 시간당 단위를 보고 달러로 바꿔 `GPU_HOURLY_USD` 에 넣는다 (Pro 100단위 ≈ $10 기준이면 1단위 ≈ $0.1).
- 서빙·평가(vLLM)는 어차피 RunPod 이다 (`serving/README.md`). 학습 결과(어댑터)는 드라이브에 저장되는지 꼭 확인한다 — 저장 실패로 어댑터를 잃은 적이 있다.

In [ ]:
import json
import math
import subprocess

TRIAL_STEPS = 20
for mode, flag in (("lora", ""), ("qlora", "--load-in-4bit")):
    trial = f"/content/trial-{TASK}-{mode}"
    command = (
        f"python training/tagging/train_lora.py {COMMON_ARGS} --output-dir {trial} --max-steps {TRIAL_STEPS} {flag}"
    )
    subprocess.run(command, shell=True, check=True)

manifest = json.loads(open(f"{DATA_DIR}/manifest.json").read())
batch, accum = (int(COMMON_ARGS.split(f"--{k} ")[1].split()[0]) for k in ("batch-size", "grad-accum"))
epochs = float(COMMON_ARGS.split("--epochs ")[1].split()[0])
total_steps = math.ceil(manifest["counts"]["train"] / (batch * accum)) * epochs
for mode in ("lora", "qlora"):
    m = json.loads(open(f"/content/trial-{TASK}-{mode}/run_metrics.json").read())
    hours = m["train_seconds"] / TRIAL_STEPS * total_steps / 3600
    print(
        f"{mode:5s} {m['gpu']} · 최대 메모리 {m['train_peak_gib']} GiB (예약 {m['train_reserved_gib']}) · "
        f"전체 {total_steps:.0f} 스텝 예상 {hours:.1f}시간 · 예상 비용 ${hours * GPU_HOURLY_USD:.2f}"
    )

## 5. LoRA 학습 (원본 16비트) — 먼저
LoRA 를 먼저 돌리는 이유: QLoRA 가 따라가야 할 **품질 기준점**이고, 압축을 풀며 계산하지 않아 더 빠르다.
`!python` 으로 따로 실행하므로 끝나면 GPU 메모리가 비워진다. 런타임을 다시 시작하지 않아도 된다.

In [ ]:
RUN = f"{OUT_ROOT}/{TASK}-lora-s{SEED}"
!python training/tagging/train_lora.py {COMMON_ARGS} --output-dir {RUN}

## 6. QLoRA 학습 (원본 4비트 NF4) — 5번과 인자가 같고 `--load-in-4bit` 만 다르다

In [ ]:
RUN = f"{OUT_ROOT}/{TASK}-qlora-s{SEED}"
!python training/tagging/train_lora.py {COMMON_ARGS} --output-dir {RUN} --load-in-4bit

## 7. 비교표

| 칸 | 뜻 |
|---|---|
| `train_peak_gib` | 학습 중 실제로 쓴 최대 GPU 메모리 |
| `train_reserved_gib` | PyTorch 가 잡아 둔 메모리 (`nvidia-smi` 값에 가까움) |
| `train_seconds`, `train_cost_usd` | 학습 시간, 시간 × `GPU_HOURLY_USD` |
| `eval_loss` | 검증 손실. 품질 판정은 7번의 평가 점수로 한다 |

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for mode in ("lora", "qlora"):
    path = Path(OUT_ROOT) / f"{TASK}-{mode}-s{SEED}" / "run_metrics.json"
    if path.exists():
        rows.append(json.loads(path.read_text()))
    else:
        print("아직 없음:", path)
columns = [
    "mode",
    "gpu",
    "dtype",
    "train_peak_gib",
    "train_reserved_gib",
    "train_seconds",
    "train_cost_usd",
    "eval_loss",
    "adapter_mb",
]
display(pd.DataFrame(rows)[columns] if rows else "기록 없음")

## 8. 다음 단계 — 품질 점수 (`training/RESULTS.md` 의 "갱신하는 법")

비용만으로는 고르지 않는다. 두 어댑터를 **같은 서빙 조건**(같은 원본 정밀도, greedy)에 올려 품질을 잰다.

1. RunPod 에서 어댑터들을 한 서버에 올린다. 평가할 때는 `--override-generation-config '{"temperature": 0}'` 로 greedy 를 맞춘다
   (앱이 temperature 를 보내지 않아 그냥 띄우면 temperature 1.0 샘플링이다).
2. 태깅 `scripts/eval_tagging.py --slm --model <어댑터 이름>`, 추출 `scripts/eval_extraction.py --slm --model <어댑터 이름>` (regression)
3. 결과 JSON 의 `summary` 를 `training/RESULTS.md` 표에 옮긴다. 게이트(F2 85%, F3 100%·80%·5%) 통과 여부도 `summary.gate` 에 있다.

판정(제안): 품질 차이가 오차 범위(태깅 47건이라 몇 건) 안이면 더 싼 쪽, 밖이면 품질이 좋은 쪽을 고르되 그 차이가 비용 차이만큼 가치 있는지 표로 남긴다.